# 365 Probabilidades - Dia #109
## Qual a probabilidade de o elevador chegar no sentido errado?

**Tipo:** Estrutural
**Data de publicação:** 2026-09-30
**Ferramenta:** Python
**Decisão analisada:** Vale a pena esperar, ou é melhor descer pela escada?
**Hashtag:** #365Probabilidades #Dia109

---

### 📖 A História

Você aperta o botão para descer. O elevador chega. E está subindo.

Não é impressão sua, não é azar, e não é o prédio implicando com você. É uma
consequência aritmética de onde o elevador passa a maior parte do tempo, e foi descrita
formalmente em 1958.

O curioso é que a explicação clássica, a que aparece em livro de curiosidade matemática
e em vídeo de internet, está certa sobre o mecanismo e errada sobre quem sofre mais.
Ela diz que o problema é dos andares de cima. Simulando um prédio de verdade, o
problema é de todo mundo, e é pior embaixo.

Este é o segundo dia da série dos números cabalísticos: crenças e números que todo mundo
repete, testados com conta própria.

---

### 📚 O Conceito: o paradoxo do elevador

Em 1958, George Gamow e Marvin Stern publicaram o problema num livro de recreação
matemática. Gamow trabalhava num andar alto e reclamava que o elevador quase sempre
chegava subindo. Stern trabalhava num andar baixo e reclamava do contrário.

O raciocínio deles é geométrico e simples. Se o elevador estiver numa posição qualquer
do prédio, igualmente provável em qualquer altura, a chance de ele estar **abaixo** de
você é a fração do prédio que fica abaixo de você. Quando ele vier, virá subindo.

Para quem está no andar k de um prédio de n andares e quer **descer**, a chance de o
elevador chegar no sentido errado é:

**P(sentido errado) = (k − 1) / (n − 1)**

Quanto mais alto o andar, maior a chance. No último andar, 100%.

Essa é a resposta clássica. Ela supõe uma coisa que nenhum prédio cumpre: que o
elevador esteja em posição uniforme.

---

### 🧮 O Modelo

Três camadas, da fórmula fechada até o prédio de verdade.

1. **A fórmula exata de Gamow e Stern**, aplicada a três prédios.
2. **Vários elevadores.** Se o problema é a posição de um carro, quatro carros deveriam
   resolver. Simulação com semente declarada mostra quanto resolvem de fato.
3. **Simulação de eventos discretos do prédio real.** A premissa da posição uniforme cai.
   Elevador ocioso volta ao térreo e fica lá a maior parte do tempo. Cada viagem sai de
   onde ele está e vai para onde chamaram. Registramos o sentido do carro toda vez que
   ele cruza o andar de quem espera.

**Fontes:**
- Gamow, G. & Stern, M. (1958). *Puzzle-Math*. Viking Press. Origem do paradoxo do
  elevador e da fórmula geométrica.
- Knuth, D. E. (1969). "The Gamow-Stern Elevator Problem". *Journal of Recreational
  Mathematics* 2, 131-137. Tratamento do caso de vários elevadores.
  **Artigo não consultado nesta produção.** Citado como existência da literatura, e
  nenhum número deste notebook vem dele.

**Nota metodológica:** o fator ×0,80 não se aplica. Não há proporção de survey aqui.
A camada 1 é geometria fechada e as camadas 2 e 3 são simulações próprias, com
parâmetros e semente declarados.

**Ponte interna:** o **#103** tratou de escolher a fila errada no supermercado. Lá a
armadilha era a informação que falta na hora de escolher. Aqui não há escolha nenhuma:
a armadilha está na posição de repouso do sistema.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEMENTE = 42
VIAGENS = 400_000

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'

def mil(n):
    return f"{int(n):,}".replace(",", ".")

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

print("✅ Bibliotecas carregadas · semente", SEMENTE)

In [ ]:
# --- OS PRÉDIOS ---
# Térreo = andar 1. O observador está no andar k e quer DESCER.

PREDIOS = {
    'O prédio do Gamow':      dict(n=7,  elevadores=1, andares=[2, 4, 6]),
    'Residencial de 12':      dict(n=12, elevadores=2, andares=[2, 4, 10]),
    'Torre de 20':            dict(n=20, elevadores=4, andares=[5, 10, 18]),
}

# Fração do tempo que o elevador ocioso passa parado no térreo.
# Escolha nossa, declarada, com análise de sensibilidade no Passo 3.
P_TERREO = 0.90

# Gamow & Stern: P(sentido errado) para quem esta no andar k e quer descer
def geometria(k, n):
    return (k - 1) / (n - 1)

print("=" * 70)
print("  OS TRÊS PRÉDIOS · A FÓRMULA DE GAMOW E STERN")
print("=" * 70)
print("\n  P(sentido errado) = (k − 1) / (n − 1), para quem está no andar k e quer descer\n")
for nome, p in PREDIOS.items():
    n = p['n']
    linha = " · ".join(f"andar {k}: {vir(geometria(k, n) * 100)}%" for k in p['andares'])
    print(f"  {nome:<22} ({n} andares, {p['elevadores']} elevador"
          f"{'es' if p['elevadores'] > 1 else ''})")
    print(f"  {'':22} {linha}")
print(f"\n  No último andar a fórmula dá 100%: o elevador nunca pode vir de cima.")
print(f"  No primeiro andar acima do térreo, ela dá quase zero.")
print(f"\n  Fração do tempo no térreo usada na camada 3: {P_TERREO * 100:.0f}%")
print("  ×0,80: não se aplica. Geometria fechada e simulação própria.")
print("=" * 70)

In [ ]:
# --- O MODELO ---

rng = np.random.default_rng(SEMENTE)

# ---------------------------------------------------------------------------
# CAMADA 2 — vários elevadores em posições independentes, o primeiro que chegar
def varios(n, k, m, reps=400_000, semente=SEMENTE):
    g = np.random.default_rng(semente)
    pos = g.uniform(1, n, size=(reps, m))
    mais_perto = pos[np.arange(reps), np.abs(pos - k).argmin(1)]
    return (mais_perto < k).mean()          # veio de baixo = chega subindo

# ---------------------------------------------------------------------------
# CAMADA 3 — eventos discretos do prédio real
# A origem de cada viagem é o térreo (elevador ocioso voltou) ou o destino da
# viagem anterior, que é uniforme. Isso torna a origem uniforme com prob (1−p),
# o que permite vetorizar sem perder o mecanismo.
def predio_real(n, k, p_terreo, viagens=VIAGENS, semente=SEMENTE):
    g = np.random.default_rng(semente)
    origem = np.where(g.random(viagens) < p_terreo, 1, g.integers(1, n + 1, viagens))
    destino = g.integers(1, n + 1, viagens)
    vale = origem != destino
    o, d = origem[vale], destino[vale]
    cruza = (np.minimum(o, d) <= k) & (k <= np.maximum(o, d))
    return (d > o)[cruza].mean()            # subindo = sentido errado

print("=" * 70)
print("  CAMADA 2 · VÁRIOS ELEVADORES RESOLVEM?")
print("=" * 70)
print(f"\n  Posições independentes e uniformes, o primeiro a chegar (semente {SEMENTE})\n")
for nome, p in PREDIOS.items():
    n, k = p['n'], p['andares'][-1]
    print(f"  {nome} · andar {k}")
    for m in [1, 2, 4]:
        print(f"    {m} elevador{'es' if m > 1 else ' ':<3} → sentido errado em "
              f"{vir(varios(n, k, m) * 100)}%")
print("\n  Dobrar o número de carros não corta o problema pela metade.")

print("\n" + "=" * 70)
print("  CAMADA 3 · O PRÉDIO REAL")
print("=" * 70)
print(f"\n  {mil(VIAGENS)} viagens, semente {SEMENTE}, elevador ocioso volta ao térreo")
print(f"  e fica lá {P_TERREO*100:.0f}% do tempo\n")
print("  andar | a geometria diz | o prédio real dá | diferença")
print("  " + "-" * 56)
N_REF = 12
inversao = []
for k in range(2, N_REF + 1):
    g_, r_ = geometria(k, N_REF) * 100, predio_real(N_REF, k, P_TERREO) * 100
    inversao.append((k, g_, r_))
    dif = ("+" if r_ >= g_ else "") + vir(r_ - g_)
    print(f"   {k:>4}  |     {vir(g_):>6}%     |     {vir(r_):>6}%     | {dif:>8} pp")
print("  " + "-" * 56)
print(f"\n  A geometria SOBE do andar 2 ({vir(inversao[0][1])}%) ao último "
      f"({vir(inversao[-1][1])}%).")
print(f"  O prédio real DESCE do andar 2 ({vir(inversao[0][2])}%) ao último "
      f"({vir(inversao[-1][2])}%).")
print(f"  As duas curvas apontam para lados opostos, e nenhum andar fica abaixo de "
      f"{vir(min(r for _, _, r in inversao))}%.")
print(f"\n  O andar que a fórmula elege como o MELHOR lugar do prédio (o 2, com "
      f"{vir(inversao[0][1])}%)")
print(f"  é, na simulação, o PIOR de todos ({vir(inversao[0][2])}%).")

# ---------------------------------------------------------------------------
# SENSIBILIDADE — o resultado depende de quanto o elevador fica no térreo?
print("\n" + "=" * 70)
print("  SENSIBILIDADE · E SE O ELEVADOR NÃO MORASSE NO TÉRREO?")
print("=" * 70)
print("\n  % do tempo no térreo | andar 2 | andar 4 | andar 10")
print("  " + "-" * 52)
for p in [0.0, 0.25, 0.50, 0.75, 0.90, 1.0]:
    vals = [predio_real(N_REF, k, p) * 100 for k in (2, 4, 10)]
    print(f"        {p*100:>5.0f}%          | " +
          " | ".join(f"{vir(v):>6}%" for v in vals))
print("  " + "-" * 52)
print("  Com posição uniforme (0% no térreo) todo andar dá 50%, e não o que a")
print("  geometria de Gamow prevê. São perguntas diferentes: a fórmula descreve")
print("  a posição do carro num instante, a simulação descreve o sentido dele")
print("  nas vezes em que ele de fato cruza o seu andar.")
print("=" * 70)

In [ ]:
# --- VISUALIZAÇÃO ---

# GRÁFICO 1 — a fórmula de Gamow nos três prédios
fig1, ax1 = plt.subplots(figsize=(12, 8))
cores = [VERDE, DOURADO, VERMELHO]
for cor, (nome, p) in zip(cores, PREDIOS.items()):
    n = p['n']
    ks = np.arange(2, n + 1)
    ax1.plot(ks, [geometria(k, n) * 100 for k in ks], marker='o', markersize=6,
             linewidth=2.4, color=cor, label=f'{nome} ({n} andares)')
ax1.axhline(50, color=CINZA, linestyle=':', linewidth=1.4)
ax1.text(2.2, 52, 'meio a meio', fontsize=10, color=CINZA)
ax1.set_xlabel('Andar de quem espera', fontsize=12)
ax1.set_ylabel('P(o elevador chega subindo) %', fontsize=12)
ax1.set_ylim(0, 105)
ax1.set_title('A resposta clássica: quanto mais alto, pior\n'
              'P(sentido errado) = (k − 1) / (n − 1), para quem quer descer',
              fontsize=14, pad=16)
ax1.legend(frameon=False, fontsize=11, loc='lower right')
plt.figtext(0.5, 0.01, 'Fonte: Gamow & Stern, Puzzle-Math, 1958 | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-109-grafico-01-a-formula.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 — vários elevadores
fig2, ax2 = plt.subplots(figsize=(12, 8))
larg, ms = 0.26, [1, 2, 4]
rotulos, x = [], np.arange(len(PREDIOS))
dados = {m: [] for m in ms}
for nome, p in PREDIOS.items():
    n, k = p['n'], p['andares'][-1]
    rotulos.append(f'{nome}\n{n} andares, andar {k}')
    for m in ms:
        dados[m].append(varios(n, k, m) * 100)
for i, (m, cor) in enumerate(zip(ms, [VERMELHO, DOURADO, VERDE])):
    b = ax2.bar(x + (i - 1) * larg, dados[m], larg, color=cor, alpha=0.88,
                label=f'{m} elevador' + ('es' if m > 1 else ''), zorder=2)
    for r, v in zip(b, dados[m]):
        ax2.text(r.get_x() + r.get_width() / 2, v + 1.2, f'{vir(v)}%',
                 ha='center', fontsize=10.5, fontweight='bold')
ax2.axhline(50, color=CINZA, linestyle='--', linewidth=1.6, zorder=3)
ax2.text(-0.45, 51.5, 'meio a meio', fontsize=10.5, color=CINZA)
ax2.set_xticks(x); ax2.set_xticklabels(rotulos, fontsize=11)
ax2.set_ylabel('P(o elevador chega subindo) %', fontsize=12)
ax2.set_ylim(0, 100)
ax2.set_title('Mais elevadores ajudam, e menos do que você esperaria\n'
              f'Posições independentes, o primeiro a chegar (semente {SEMENTE})',
              fontsize=14, pad=16)
ax2.legend(frameon=False, fontsize=11)
plt.figtext(0.5, 0.01, f'Simulação própria, semente {SEMENTE} | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-109-grafico-02-varios-elevadores.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 — A INVERSÃO
fig3, ax3 = plt.subplots(figsize=(12, 8))
ks = [k for k, _, _ in inversao]
gs = [g for _, g, _ in inversao]
rs = [r for _, _, r in inversao]
ax3.plot(ks, gs, marker='o', markersize=8, linewidth=3, color=DOURADO, zorder=3,
         label='O que a fórmula prevê')
ax3.plot(ks, rs, marker='s', markersize=8, linewidth=3, color=VERMELHO, zorder=3,
         label=f'O que o prédio real dá ({P_TERREO*100:.0f}% do tempo no térreo)')
ax3.fill_between(ks, gs, rs, color=VERMELHO, alpha=0.07, zorder=1)
ax3.annotate(f'a fórmula diz que o andar 2\né o melhor lugar: {vir(gs[0])}%',
             xy=(2, gs[0]), xytext=(5.6, 7), fontsize=11.5, color=DOURADO, ha='left',
             arrowprops=dict(arrowstyle='->', color=DOURADO, lw=1.6))
ax3.annotate(f'e a simulação diz que é o pior: {vir(rs[0])}%',
             xy=(2, rs[0]), xytext=(5.6, 72), fontsize=11.5, color=VERMELHO, ha='left',
             arrowprops=dict(arrowstyle='->', color=VERMELHO, lw=1.6))
ax3.set_xticks(ks)
ax3.set_xlabel('Andar de quem espera (prédio de 12 andares)', fontsize=12)
ax3.set_ylabel('P(o elevador chega subindo) %', fontsize=12)
ax3.set_ylim(0, 105)
ax3.set_title('As duas respostas apontam para lados opostos\n'
              'A geometria sobe com o andar. O prédio real desce, e não solta ninguém.',
              fontsize=14, pad=16)
ax3.legend(frameon=False, fontsize=11.5, loc='lower right')
plt.figtext(0.5, 0.01, f'Gamow & Stern 1958 contra simulação própria de '
            f'{mil(VIAGENS)} viagens, semente {SEMENTE} | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-109-grafico-03-a-inversao.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 — o peso do térreo
fig4, ax4 = plt.subplots(figsize=(12, 8))
ps = np.linspace(0, 1, 21)
for cor, k in zip([VERMELHO, DOURADO, VERDE], [2, 4, 10]):
    ax4.plot(ps * 100, [predio_real(N_REF, k, p) * 100 for p in ps],
             marker='o', markersize=5, linewidth=2.4, color=cor, label=f'andar {k}')
ax4.axhline(50, color=CINZA, linestyle=':', linewidth=1.4)
ax4.axvline(P_TERREO * 100, color=CINZA, linestyle='--', linewidth=1.6)
ax4.text(P_TERREO * 100 - 2, 20, 'o valor usado\nno modelo', ha='right', fontsize=10.5,
         color=CINZA)
ax4.set_xlabel('Fração do tempo em que o elevador está parado no térreo (%)', fontsize=12)
ax4.set_ylabel('P(o elevador chega subindo) %', fontsize=12)
ax4.set_ylim(40, 102)
ax4.set_title('Não é o prédio, é o lugar onde o elevador descansa\n'
              'Prédio de 12 andares, para quem quer descer', fontsize=14, pad=16)
ax4.legend(frameon=False, fontsize=11.5)
plt.figtext(0.5, 0.01, f'Simulação própria, semente {SEMENTE} | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-109-grafico-04-o-terreo.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

**Não é azar. É onde o elevador dorme.**

A explicação clássica, a de Gamow e Stern, é elegante e está meio certa. Ela diz que o
elevador quase sempre vem do lado do prédio que tem mais andares, e que por isso quem
mora em cima sofre. A fórmula é limpa: no andar 2 de um prédio de 12, a chance de ele
chegar no sentido errado seria de apenas 9,1%.

Só que a fórmula supõe que o elevador esteja em posição sorteada, igualmente provável em
qualquer altura. Nenhum elevador se comporta assim. Ele volta ao térreo e fica lá.

Simulando 400 mil viagens de um prédio que se comporta desse jeito, **o andar 2 passa de
9,1% para 98,3%**. Não é um ajuste. É o contrário do que a fórmula prometia.

E a inversão é completa. A geometria sobe conforme você sobe, de 9,1% até 100%. A
simulação desce conforme você sobe, de 98,3% até 91,2%. **Nenhum andar do prédio fica
abaixo de 91%.**

O motivo cabe numa frase: se o elevador mora no térreo, quase toda vez que ele cruza o
seu andar é porque alguém acima de você chamou, e nessa hora ele está subindo. Quanto
mais baixo você mora, mais gente tem acima chamando, e mais vezes ele passa por você no
sentido errado.

Mais elevadores ajudam menos do que parece. No andar 10 do prédio de 12, com um carro em
posição sorteada a chance de sentido errado é de 81,9%. Com dois carros, 70,2%. Com
quatro, 58,2%. Quadruplicar a frota não chega a empatar o jogo.

A pergunta que fica:

*Quantas coisas que você chama de azar são só a posição de repouso de um sistema que
você nunca desenhou?*

---

### ⚠️ Limitações do Modelo

- **A camada 3 é simulação, não medida.** Nenhum prédio real foi observado. Os
  parâmetros (fração do tempo no térreo, chamadas uniformes entre os andares) são
  escolhas declaradas, com semente 42, e a análise de sensibilidade mostra o que muda
  quando a fração do térreo varia de 0% a 100%.
- **As duas camadas respondem a perguntas diferentes, e isso precisa estar claro.** A
  fórmula de Gamow descreve a posição do carro num instante sorteado. A simulação
  descreve o sentido do carro nas vezes em que ele cruza o seu andar. Com posição
  uniforme, a simulação dá 50% em todo andar, e não o valor da fórmula. Os números não
  são comparáveis ponto a ponto; o que se compara é a **direção** das duas curvas.
- **As chamadas são uniformes entre os andares.** Num prédio residencial de verdade, o
  tráfego é concentrado em horários e o térreo é origem e destino da maioria das
  viagens, o que provavelmente tornaria o efeito ainda mais forte, não mais fraco.
- **O artigo de Knuth (1969) não foi consultado.** Ele existe, trata do caso de vários
  elevadores, e é citado como literatura. Nenhum número deste notebook vem dele, e a
  camada 2 é simulação própria.
- **O modelo ignora o tempo de espera.** Ele responde em que sentido o elevador chega,
  não quanto tempo você espera, que é outra pergunta e pede outra ferramenta.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*